In [ ]:
# Analyze pymatgen usage of dependency APIs and generate sankey diagram

In [ ]:
import os
from pathlib import Path

PMG_REPO_PATH = os.getenv("PMG_REPO_PATH")
PMG_COMMIT: str = "ab34799d8ab5dee80756489cf2ca28a97de78121"

# Install api-analyzer
nb_dir = Path.cwd()

api_analyzer_path = nb_dir / "api-analyzer"
!uv pip install -e "{api_analyzer_path}"

Using Python 3.12.11 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Resolved 1 package in 12ms                                           
   Building api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/figs/pmg-ap
   Building api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/figs/pmg-ap
   Building api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/figs/pmg-ap
      Built api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/figs/pmg-ap
Prepared 1 package in 541ms                                              
Uninstalled 1 package in 0.69ms
Installed 1 package in 0.94msrom file:///Users/yang/develope
 ~ api-analyzer==0.1.0 (from file:///Users/yang/developer/pymatgen-2-paper/figs/pmg-api-usage-dependent-dependency/api-analyzer)


In [ ]:
# Get required and optional dependencies
from pathlib import Path
import tomllib
from packaging.requirements import Requirement


def get_all_dependencies(pyproject_path) -> dict[str, list[str] | dict[str, list[str]]]:
    """
    Load dependencies from pyproject.toml and strip versions/markers.
    """

    def _strip_versions(reqs: list[str]) -> list[str]:
        """Return only the package names (no version/markers/extras)."""
        names = []
        for r in reqs:
            names.append(Requirement(r).name)

        return names

    data = tomllib.loads(Path(pyproject_path).read_text(encoding="utf-8"))
    project = data.get("project", {})

    required = project.get("dependencies", [])
    optional = project.get("optional-dependencies", {})

    return {
        "required": _strip_versions(required),
        "optional": {k: _strip_versions(v) for k, v in optional.items()},
    }


deps = get_all_dependencies(f"{PMG_REPO_PATH}/pyproject.toml")
print("Required:", deps["required"])
print("Optional groups:", list(deps["optional"].keys()))

Required: ['bibtexparser', 'joblib', 'matplotlib', 'monty', 'networkx', 'numpy', 'orjson', 'palettable', 'pandas', 'plotly', 'requests', 'ruamel.yaml', 'scipy', 'scipy', 'spglib', 'sympy', 'tabulate', 'tqdm', 'uncertainties']
Optional groups: ['abinit', 'ase', 'electronic_structure', 'matcalc', 'mlp', 'numba', 'numpy-v1', 'optional', 'prototypes', 'symmetry', 'tblite', 'vis', 'zeopp']


In [ ]:
# Get top-level subpackages of pymatgen
import pkgutil
import pymatgen


def list_top_level_subpackages(pkg) -> list[str]:
    return sorted(
        {
            name.split(".")[1]
            for _, name, _ in pkgutil.iter_modules(pkg.__path__, pkg.__name__ + ".")
        }
    )


subpacks = list_top_level_subpackages(pymatgen)
for pack in ["dao", "util", "optimization", "apps", "cli", "command_line", "vis"]:
    subpacks.remove(pack)

print(subpacks)

['alchemy', 'core', 'electronic_structure', 'entries', 'phonon', 'symmetry', 'transformations']


In [ ]:
# Analyze each required dependencies' API usage by each pmg subpackage

OUTPUT_YAML_FILE: str = "pmg_dependencies_usage.yaml"

import yaml
from pathlib import Path

from api_analyzer import analyze_paths

results: dict[str, dict[str, dict[str, dict[str, int]]]] = {}

for dep in deps["required"]:
    results[dep] = {}
    for pack in subpacks:
        path = Path(PMG_REPO_PATH) / "src" / "pymatgen" / pack
        aliases, usage = analyze_paths(path, package=dep)

        results[dep][pack] = {
            "aliases": aliases,
            "usage": usage,
        }

output_file = Path(OUTPUT_YAML_FILE)
with output_file.open("w", encoding="utf-8") as f:
    yaml.safe_dump(results, f, sort_keys=False, indent=2)

print(f"Saved results to {output_file}")

Saved results to pmg_dependencies_usage.yaml


In [ ]:
# Plot a Sankey diagram showing pymatgen module usage of 3rd-party dependencies.

import yaml
import plotly.graph_objects as go


def plot_pmg_dependency_sankey(data: dict, title):
    # Collect total usage counts per (dependency, module)
    links = []
    dep_totals: dict[str, int] = {}

    for dep, modules in data.items():
        total_dep_usage = 0
        for module, mod_info in modules.items():
            usage = mod_info.get("usage", {}) or {}
            total_usage = sum(usage.values())
            if total_usage > 0:
                links.append((dep, f"pymatgen.{module}", total_usage))
                total_dep_usage += total_usage
        dep_totals[dep] = total_dep_usage

    if not links:
        raise ValueError("No usage data found in YAML.")

    # Sort dependencies by total usage (descending)
    # TODO: it's not really working
    sorted_deps = [d for d, _ in sorted(dep_totals.items(), key=lambda x: -x[1])]
    sorted_modules = sorted({mod for _, mod, _ in links})
    labels = sorted_deps + sorted_modules
    label_to_idx = {lbl: i for i, lbl in enumerate(labels)}

    # Build link data
    sources = [label_to_idx[dep] for dep, _, _ in links]
    targets = [label_to_idx[mod] for _, mod, _ in links]
    values = [v for _, _, v in links]

    fig = go.Figure(
        go.Sankey(
            arrangement="snap",
            node=dict(
                label=labels,
                pad=20,
                thickness=20,
                color=[
                    "#8FB9A8" if label in sorted_deps else "#F7B267" for label in labels
                ],
            ),
            link=dict(source=sources, target=targets, value=values),
        )
    )

    fig.update_layout(
        title_text=title,
        font=dict(size=12),
        height=700,
        width=1000,
    )

    fig.show()

    fig.write_image(str("pmg_dependency_usage.svg"), format="svg", scale=2)


with open(OUTPUT_YAML_FILE) as f:
    data = yaml.safe_load(f)

plot_pmg_dependency_sankey(data, title="pymatgen 3rd-party dependency usage")